# Download C4 Data
Downloads a slice of allenai/c4 (English) and saves as .txt files into `data_c4/`.
Run `consolidate_data` -> `tokenizer_pipeline` -> `pre_train` after this.

In [ ]:
!pip install -q datasets

In [ ]:
import os
from google.colab import drive
from datasets import load_dataset

drive.mount('/content/drive')

OUT_DIR = "/content/drive/MyDrive/sparkyllm/datasets_pretrain/data_c4"
os.makedirs(OUT_DIR, exist_ok=True)

MAX_DOCS = 1_000_000
DOCS_PER_FILE = 10_000
SEPARATOR = "\n<|endoftext|>\n"

print(f"Streaming C4 English, downloading {MAX_DOCS:,} documents...")
ds = load_dataset("allenai/c4", "en", split="train", streaming=True)

buf = []
file_idx = 0
total_chars = 0

for i, example in enumerate(ds):
    if i >= MAX_DOCS:
        break
    buf.append(example["text"])
    if len(buf) >= DOCS_PER_FILE:
        path = os.path.join(OUT_DIR, f"c4_part_{file_idx:03d}.txt")
        content = SEPARATOR.join(buf)
        with open(path, "w", encoding="utf-8") as f:
            f.write(content)
        size_mb = os.path.getsize(path) / 1024 / 1024
        total_chars += len(content)
        print(f"  {path.split('/')[-1]}: {size_mb:.1f} MB ({len(buf)} docs)")
        buf = []
        file_idx += 1

    if (i + 1) % 50_000 == 0:
        print(f"  ... {i + 1:,} docs processed")

# Write remainder
if buf:
    path = os.path.join(OUT_DIR, f"c4_part_{file_idx:03d}.txt")
    content = SEPARATOR.join(buf)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    total_chars += len(content)
    size_mb = os.path.getsize(path) / 1024 / 1024
    print(f"  {path.split('/')[-1]}: {size_mb:.1f} MB ({len(buf)} docs)")
    file_idx += 1

total_mb = total_chars / 1024 / 1024
print(f"\nDone!")
print(f"  {i + 1:,} documents in {file_idx} files")
print(f"  Total: ~{total_mb:.0f} MB")
print(f"  Saved to: {OUT_DIR}")
print(f"\nNext: run consolidate_data -> tokenizer_pipeline (TRAIN_NEW_TOKENIZER=True) -> pre_train")